![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 14 -- Competition: Outcome After Treatment (starter)

A hospital wrote down facts about each patient when treatment started (age, the size and stage of the illness, lab tests) and later wrote down how it ended. You get the facts for every patient; for the training patients you also get the ending. Predict, for every test patient, whether the ending is the worse outcome (1) or not (0).

**Your role:** start from this notebook. Keep the two cells marked *do not modify* as they are, change anything else, and upload the `submission.csv` this notebook writes to the Kaggle page. Hand in your final notebook through the Day 14 form.

**Dataset:** `train.csv` (2,092 patients with the `outcome` column), `test.csv` (1,127 patients without it) and `sample_submission.csv` (the exact shape of the file you upload). The column table is in the Kaggle Data tab, in the competition README and in the cell below.

## Rules in short

- Teams of at most 3 (solo is fine); 12 submissions per team per Kaggle day (the counter resets at 03:00 Riyadh time); choose your 2 final submissions before the end.
- The score is **F1 on outcome = 1**. The public leaderboard uses 40% of the test patients; the final ranking uses the other 60%.
- Allowed: pandas, numpy, scikit-learn, matplotlib (anything taught in Week 2). No other data, no searching for where this table comes from.
- Upload your final notebook through the Day 14 Google Form by Saturday 13:30 Riyadh time, one per team.

---
# Setup

In [ ]:
# pandas, numpy, scikit-learn and matplotlib are preinstalled on Kaggle and Colab. kagglehub is only needed outside Kaggle.
# !pip install -q kagglehub

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
#  DATA LOADING -- do not modify
# ============================================================
DATA_ID = "sattamjaltwaim/ai-olympiad-l1-2027-week-2-competition-data"          # the Kaggle dataset of this competition


def find_data():
    local = os.environ.get("COMP_DATA")                      # a local folder (instructor's check)
    if local and os.path.exists(os.path.join(local, "train.csv")):
        return local
    for folder in sorted(glob.glob("/kaggle/input/*")):      # inside a Kaggle notebook
        if os.path.exists(os.path.join(folder, "train.csv")):
            return folder
    import kagglehub                                         # Colab or your own computer
    return kagglehub.dataset_download(DATA_ID)


DATA = find_data()
train = pd.read_csv(os.path.join(DATA, "train.csv"))
test = pd.read_csv(os.path.join(DATA, "test.csv"))
sample = pd.read_csv(os.path.join(DATA, "sample_submission.csv"))
print("train", train.shape, "test", test.shape, "sample", sample.shape)
print("share of outcome == 1 in train:", round(train["outcome"].mean(), 3))

## The columns, in plain words

Every row is one patient. The table is as the hospital collected it: some cells are blank and some numbers are impossible (a size below zero, a count below zero). What usually goes together: a bigger growth, more affected samples, a higher stage or more spread go with outcome 1 more often; a positive lab test goes with outcome 0 more often. Nothing else about medicine is needed.

| column | what it is |
|---|---|
| `id` | row number, not a feature |
| `age` | the patient's age in years, written with decimals as recorded |
| `population_group` | which of three groups of patients the hospital uses: A, B or C (the groups have no order) |
| `living_situation` | who the patient lives with: H1 married, H2 single, H3 divorced, H4 widowed, H5 separated |
| `growth_size_class` | how big the growth was when it was found, in four classes from A1 (smallest) to A4 (largest) |
| `spread_class` | how far the illness had reached into the tissue next to the growth, from B1 (least) to B3 (most) |
| `size_and_spread` | the two classes above written together, for example A2_B1 |
| `overall_stage` | the doctors' overall stage, from C1 (earliest) to C5 (most advanced); blank for some patients |
| `extent` | E1: the illness stayed in one area; E2: it had reached distant parts of the body |
| `cell_appearance` | how the cells look under a microscope: K1 almost normal, K2 somewhat changed, K3 very changed, K4 nothing like normal cells |
| `cell_grade` | the same idea as a number written in text: 1, 2 or 3 (one odd value appears) |
| `growth_size_mm` | the size of the growth in millimetres; blank for some patients, and some values are impossible for a size |
| `test_e` | a lab test, P (positive) or N (negative); a positive test means a common treatment is expected to work |
| `test_p` | a second lab test of the same kind, P or N |
| `tests_e_and_p` | the two test results written together, for example P_N |
| `samples_checked` | how many small tissue samples the doctors checked |
| `samples_affected` | how many of those samples showed the illness |
| `samples_clear` | how many of those samples did not show it |
| `blood_pressure_high` | blood pressure, the upper number |
| `blood_pressure_low` | blood pressure, the lower number |
| `cholesterol` | cholesterol level in the blood |
| `body_temperature` | body temperature in degrees Celsius |
| `blood_oxygen` | oxygen level in the blood, in percent |
| `breaths_per_minute` | breaths per minute |
| `blood_sugar` | sugar level in the blood |
| `bmi` | body mass index: weight compared with height |
| `pulse` | heart beats per minute |
| `creatinine` | a blood test value that doctors use to check the kidneys |
| `uric_acid` | a blood test value |
| `hemoglobin` | the part of the blood that carries oxygen |
| `kidney_filter_rate` | how fast the kidneys filter the blood |
| `sodium` | salt (sodium) level in the blood |
| `potassium` | potassium level in the blood |
| `albumin` | a protein level in the blood |
| `lactate` | a blood test value |
| `outcome` | 0 = the patient was doing well at the end of the follow-up; 1 = the worse outcome, the patient did not recover (about one patient in six); only in train.csv |


---
# Look at the table first

The Day 06 routine: shape, types, blanks, `describe()`, and `value_counts()` on the text columns. Read every line of the output before modelling; the table has surprises.

In [ ]:
train.head()

In [ ]:
print(train.dtypes.value_counts())
print()
print(train.isna().sum()[train.isna().sum() > 0])

In [ ]:
train.describe().T.round(2)

In [ ]:
for col in ["living_situation", "cell_grade", "overall_stage"]:
    print(train[col].value_counts(dropna=False).to_dict(), "\n")

---
# Baseline: a logistic regression that beats the two trivial answers

Predicting 1 for everyone scores F1 0.27 and predicting 0 for everyone scores 0.00. A logistic regression on the raw columns with `class_weight="balanced"` (Day 11) scores about 0.37. This cell is the floor; everything you learned this week is for beating it.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = train.drop(columns=["id", "outcome"])
y = train["outcome"]
X_test = test.drop(columns=["id"])
num_cols = X.select_dtypes("number").columns
txt_cols = X.select_dtypes("object").columns

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("txt", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), txt_cols),
])
model = make_pipeline(prep, LogisticRegression(max_iter=3000, class_weight="balanced"))

cv = StratifiedKFold(5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="f1")
print("5-fold F1:", scores.round(3), "  mean", round(scores.mean(), 3), "  spread", round(scores.std(), 3))

model.fit(X, y)
test_pred = model.predict(X_test)

---
# Write the submission

Pass an array of 0/1 labels, one per test row, in the order of `test.csv`. **Do not modify** the function.

In [ ]:
# ============================================================
#  SUBMISSION GENERATOR -- do not modify
# ============================================================
def make_submission(pred, filename="submission.csv"):
    pred = np.asarray(pred).astype(int)
    assert len(pred) == len(test), f"{len(pred)} predictions for {len(test)} test rows"
    assert set(np.unique(pred)) <= {0, 1}, "predictions must be 0 or 1 (labels, not probabilities)"
    out = pd.DataFrame({"id": test["id"], "outcome": pred})
    out.to_csv(filename, index=False)
    print(f"wrote {filename}: {len(out)} rows, share of 1 = {pred.mean():.3f}")
    return out


submission = make_submission(test_pred)
submission.head()

---
# Where the points are

- **Look before you model.** `describe()` every column; a blank cell and an impossible value are both information (Day 06, Day 12).
- **The score is F1 at a threshold.** Class weights and a threshold chosen on out-of-fold predictions move it a lot; the probability cut of 0.5 is not sacred (Day 09, Day 13).
- **Not every column helps.** Permutation importance on a validation split tells you which ones the model actually uses (Day 13).
- **Trees and forests need no scaling and handle odd values differently from a line.** Compare them with the same `StratifiedKFold` (Day 10, Day 12).
- **Tune with `GridSearchCV`, then refit on all training rows.** Report the CV spread next to the mean (Day 13).
- **Blend two different models' probabilities**, then cut at the threshold you chose on out-of-fold predictions (Day 13).
- **Trust your cross-validation over the public leaderboard.** It scores 40% of the test rows; the other 60% decide.

---

Made By **Sattam Altwaim**